In [1]:
# Import libraries
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
df = pd.read_csv('../data/cleaned/cleaned_job_postings.csv')

# ---- CREATE SQLite DATABASE ----
# This creates a real database file in your project folder
conn = sqlite3.connect('../data/job_market.db')

# Load the dataframe into the database as a table called 'jobs'
df.to_sql('jobs', conn, if_exists='replace', index=False)

print("Database created successfully!")
print(f"Table 'jobs' loaded with {len(df):,} rows")

Database created successfully!
Table 'jobs' loaded with 12,217 rows


In [2]:
# ---- QUERY 1: Top 10 Job Titles ----
# This is the SQL equivalent of df['job_title'].value_counts().head(10)

query1 = """
SELECT 
    job_title,
    COUNT(*) AS job_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM jobs), 2) AS percentage
FROM jobs
GROUP BY job_title
ORDER BY job_count DESC
LIMIT 10;
"""

result1 = pd.read_sql_query(query1, conn)
print("Top 10 Job Titles:")
print(result1.to_string(index=False))

Top 10 Job Titles:
                                job_title  job_count  percentage
                     Senior Data Engineer        288        2.36
                      Senior Data Analyst        165        1.35
                            Data Engineer        150        1.23
                             Data Analyst        140        1.15
                    Senior Mlops Engineer        138        1.13
                           Data Scientist        128        1.05
                       Lead Data Engineer        123        1.01
                    Senior Data Scientist        119        0.97
                           Data Architect        115        0.94
Staff Machine Learning Engineer, Series A        101        0.83


In [3]:
# ---- QUERY 2: Top 10 Hiring Locations ----

query2 = """
SELECT 
    job_location,
    COUNT(*) AS job_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM jobs), 2) AS percentage
FROM jobs
WHERE job_location != 'Unknown'
GROUP BY job_location
ORDER BY job_count DESC
LIMIT 10;
"""

result2 = pd.read_sql_query(query2, conn)
print("Top 10 Locations:")
print(result2.to_string(index=False))

Top 10 Locations:
                   job_location  job_count  percentage
                   New York, NY        280        2.29
London, England, United Kingdom        232        1.90
                    Chicago, IL        232        1.90
              San Francisco, CA        199        1.63
                 Washington, DC        173        1.42
                     Austin, TX        169        1.38
                    Seattle, WA        164        1.34
                     Dallas, TX        158        1.29
                     Boston, MA        153        1.25
                    Atlanta, GA        151        1.24


In [4]:
# ---- QUERY 3: Top 10 Hiring Companies ----

query3 = """
SELECT 
    company,
    COUNT(*) AS job_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM jobs), 2) AS percentage
FROM jobs
GROUP BY company
ORDER BY job_count DESC
LIMIT 10;
"""

result3 = pd.read_sql_query(query3, conn)
print("Top 10 Companies:")
print(result3.to_string(index=False))

Top 10 Companies:
                  company  job_count  percentage
        Jobs For Humanity        732        5.99
  Recruiting From Scratch        390        3.19
                     Dice        193        1.58
                    Agoda        172        1.41
            Clearancejobs        166        1.36
             Clickjobs.Io        152        1.24
              Capital One         92        0.75
           Energy Jobline         73        0.60
Amazon Web Services (Aws)         72        0.59
                 Deloitte         67        0.55


In [5]:
# ---- QUERY 4: Experience Level Distribution ----

query4 = """
SELECT 
    job_level,
    COUNT(*) AS job_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM jobs), 2) AS percentage
FROM jobs
GROUP BY job_level
ORDER BY job_count DESC;
"""

result4 = pd.read_sql_query(query4, conn)
print("Experience Level Distribution:")
print(result4.to_string(index=False))

Experience Level Distribution:
 job_level  job_count  percentage
Mid-Senior      10919       89.38
 Associate       1298       10.62


In [6]:
# ---- QUERY 5: Remote vs Onsite vs Hybrid ----

query5 = """
SELECT 
    job_type,
    COUNT(*) AS job_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM jobs), 2) AS percentage
FROM jobs
GROUP BY job_type
ORDER BY job_count DESC;
"""

result5 = pd.read_sql_query(query5, conn)
print("Job Type Breakdown:")
print(result5.to_string(index=False))

Job Type Breakdown:
job_type  job_count  percentage
  Onsite      12188       99.76
  Remote         18        0.15
  Hybrid         11        0.09


In [7]:
# ---- QUERY 6: Jobs by Country ----

query6 = """
SELECT 
    country,
    COUNT(*) AS job_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM jobs), 2) AS percentage
FROM jobs
GROUP BY country
ORDER BY job_count DESC;
"""

result6 = pd.read_sql_query(query6, conn)
print("Jobs by Country:")
print(result6.to_string(index=False))

Jobs by Country:
       country  job_count  percentage
 United States       9594       78.53
United Kingdom        991        8.11
         Other        946        7.74
        Canada        625        5.12
         India         60        0.49
       Unknown          1        0.01


In [8]:
# ---- QUERY 7: Jobs Posted by Date ----

query7 = """
SELECT 
    first_seen AS date,
    COUNT(*) AS jobs_posted
FROM jobs
GROUP BY first_seen
ORDER BY first_seen ASC;
"""

result7 = pd.read_sql_query(query7, conn)
print("Jobs by Date:")
print(result7.to_string(index=False))

Jobs by Date:
      date  jobs_posted
2024-01-12         1722
2024-01-13         4946
2024-01-14         3194
2024-01-15          871
2024-01-16         1211
2024-01-17          273


In [9]:
# ---- QUERY 8: Top US Cities ----

query8 = """
SELECT 
    TRIM(SUBSTR(job_location, 1, INSTR(job_location, ',') - 1)) AS city,
    COUNT(*) AS job_count
FROM jobs
WHERE country = 'United States'
    AND job_location LIKE '%,%'
GROUP BY city
ORDER BY job_count DESC
LIMIT 10;
"""

result8 = pd.read_sql_query(query8, conn)
print("Top 10 US Cities:")
print(result8.to_string(index=False))

Top 10 US Cities:
         city  job_count
     New York        280
      Chicago        232
San Francisco        199
   Washington        173
       Austin        169
      Seattle        164
       Dallas        158
       Boston        153
      Atlanta        151
      Houston        130


In [10]:
# ---- SAVE QUERIES TO .SQL FILES ----
# This is what you'll upload to GitHub

queries = {
    '01_top_job_titles.sql': query1,
    '02_top_locations.sql': query2,
    '03_top_companies.sql': query3,
    '04_experience_levels.sql': query4,
    '05_job_types.sql': query5,
    '06_jobs_by_country.sql': query6,
    '07_jobs_by_date.sql': query7,
    '08_top_us_cities.sql': query8
}

import os
for filename, query in queries.items():
    filepath = f'../sql/{filename}'
    with open(filepath, 'w') as f:
        f.write(query.strip())
    print(f"Saved: {filename}")

print("\nAll SQL files saved to sql/ folder!")

# Close the database connection
conn.close()
print("Database connection closed.")

Saved: 01_top_job_titles.sql
Saved: 02_top_locations.sql
Saved: 03_top_companies.sql
Saved: 04_experience_levels.sql
Saved: 05_job_types.sql
Saved: 06_jobs_by_country.sql
Saved: 07_jobs_by_date.sql
Saved: 08_top_us_cities.sql

All SQL files saved to sql/ folder!
Database connection closed.
